In [19]:
import spacy

# 1. Import the model explicitly as its own Python package
import en_core_web_sm

# 2. Load it directly via the module instead of using spacy.load()
nlp = en_core_web_sm.load()

print("Model loaded successfully! You can now run your NLP tasks.")

Model loaded successfully! You can now run your NLP tasks.


In [20]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Load spaCy's English model
nlp = spacy.load("en_core_web_sm")

# Sample Data (Simulating user reviews)
data = {
    "review_id": [1, 2, 3, 4, 5, 6],
    "text": [
        "The app is constantly crashing on startup. Very frustrated.",
        "I love this app! It is clean, smooth, and works perfectly.",
        "Never received the OTP code, so I can't log in to my account.",
        "The interface looks bad and the update is crashing my phone.",
        "Security is tight, I received my verification code instantly.",
        "Great experience overall, highly recommend it."
    ],
    "sentiment": ["negative", "positive", "negative", "negative", "positive", "positive"]
}

df = pd.DataFrame(data)

# =====================================================================
# Task 1: Preprocessing (Lemmatization & Stop Word Removal)
# =====================================================================
def preprocess_text(text):
    # Process the text through the spaCy pipeline
    doc = nlp(text.lower())
    
    # Tokenize, remove stop words/punctuation, and extract lemmas
    cleaned_tokens = [
        token.lemma_ for token in doc 
        if not token.is_stop and not token.is_punct and token.text.strip()
    ]
    
    return " ".join(cleaned_tokens)

print("--- Running Preprocessing ---")
df["cleaned_text"] = df["text"].apply(preprocess_text)
print(df[["text", "cleaned_text"]])
print("\n" + "="*50 + "\n")


# =====================================================================
# Task 2: Keyword Extraction (TF-IDF by Sentiment)
# =====================================================================
print("--- Running Keyword Extraction (TF-IDF) ---")

# Combine all reviews belonging to each sentiment into single documents
sentiment_docs = df.groupby("sentiment")["cleaned_text"].apply(lambda x: " ".join(x)).reset_index()

# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(sentiment_docs["cleaned_text"])
feature_names = tfidf.get_feature_names_out()

# Extract top unique words per sentiment group
for index, row in sentiment_docs.iterrows():
    sentiment_type = row["sentiment"]
    row_vector = tfidf_matrix.getrow(index).toarray()[0]
    
    # Pair feature names with their TF-IDF scores and sort them descending
    word_scores = sorted(list(zip(feature_names, row_vector)), key=lambda x: x[1], reverse=True)
    
    print(f"Top unique words in {sentiment_type} reviews:")
    for word, score in word_scores[:4]: # Top 4 words
        print(f"  - {word}: {score:.3f}")
print("\n" + "="*50 + "\n")


# =====================================================================
# Task 3: Clustering (Thematic Grouping for 'Security/Access')
# =====================================================================
print("--- Running Targeted Clustering ---")

def assign_theme(cleaned_text):
    # Define the keywords for the cluster
    target_keywords = {"otp", "code", "receive"}
    
    # Split text into a set of unique words
    words_in_text = set(cleaned_text.split())
    
    # Check if any of our target keywords intersect with the text
    if target_keywords.intersection(words_in_text):
        return "Security/Access"
    else:
        return "Other / General"

df["theme"] = df["cleaned_text"].apply(assign_theme)
print(df[["review_id", "text", "theme"]])

--- Running Preprocessing ---
                                                text  \
0  The app is constantly crashing on startup. Ver...   
1  I love this app! It is clean, smooth, and work...   
2  Never received the OTP code, so I can't log in...   
3  The interface looks bad and the update is cras...   
4  Security is tight, I received my verification ...   
5     Great experience overall, highly recommend it.   

                                        cleaned_text  
0            app constantly crash startup frustrated  
1               love app clean smooth work perfectly  
2                       receive otp code log account  
3              interface look bad update crash phone  
4  security tight receive verification code insta...  
5          great experience overall highly recommend  


--- Running Keyword Extraction (TF-IDF) ---
Top unique words in negative reviews:
  - crash: 0.492
  - account: 0.246
  - bad: 0.246
  - constantly: 0.246
Top unique words in positive review